# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library. You'll examine its structure defined by the Croissant schema, extract tabular data, and perform initial exploratory data analysis and visualization.

### Dataset Source
The dataset is described using a Croissant schema and can be accessed at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
pd.set_option('display.max_columns', None)

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset from Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", getattr(metadata, 'identifier', ''))
print("License:", getattr(metadata, 'license', ''))

## 2. Data Overview
List all available Record Sets and Fields (`@id`s), using their Croissant `@id` identifiers. This helps understand what data tables and fields are available for extraction.

In [ ]:
# Discover record sets by their @id
print("Available record sets (by @id):")
record_sets = []
for obj in dataset.metadata._json.get('recordSet', []):
    if isinstance(obj, dict) and '@id' in obj:
        rec_id = obj['@id']
        print(f"  Record set: {rec_id}")
        record_sets.append(rec_id)

if not record_sets:
    # Attempt to search for record sets in the metadata JSON (schema may provide them inline)
    # The top-level object may have 'recordSet' with further '@id' in dicts
    # Try fetching objects with @type=cr:RecordSet
    record_sets = []
    if '@graph' in dataset.metadata._json:
        for obj in dataset.metadata._json['@graph']:
            if obj.get('@type') == 'cr:RecordSet':
                print(f"  Record set: {obj['@id']} (name: {obj.get('name', '')})")
                record_sets.append(obj['@id'])
    else:
        print("No record sets found in known locations.")

if record_sets:
    # For each record set, list fields and columns (@id)
    for rid in record_sets:
        print(f"\nFields for Record Set {rid}:")
        # Try to find the record set definition
        rec_obj = None
        if '@graph' in dataset.metadata._json:
            for obj in dataset.metadata._json['@graph']:
                if obj.get('@id') == rid:
                    rec_obj = obj
                    break
        if rec_obj is None:
            continue

        # List fields and columns by @id
        for field in rec_obj.get('field', []):
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
            print(f"  Field: {field_id}")
        for col in rec_obj.get('column', []):
            col_id = col['@id'] if isinstance(col, dict) and '@id' in col else str(col)
            print(f"  Column: {col_id}")

else:
    print('No record sets (@id) were discovered.')

## 3. Data Extraction
Load data from each record set using its `@id`. Each record set is extracted to a DataFrame, with field/column names derived from their `@id`s as found above.

In [ ]:
# Since record sets are not discovered via standard recordSet, search via @graph for cr:RecordSet
if not record_sets and '@graph' in dataset.metadata._json:
    record_sets = [obj['@id'] for obj in dataset.metadata._json['@graph'] if obj.get('@type') == 'cr:RecordSet']

# Extract all dataframes from found record sets
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from record set {record_set_id}")
            print(f"Available columns (@id): {dataframes[record_set_id].columns.tolist()}")
            print(dataframes[record_set_id].head(3))
            print('---')
        else:
            print(f"No records loaded for {record_set_id}")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

# Choose one record set with loaded data for EDA
if dataframes:
    main_rs = next(iter(dataframes))  # Use the first one loaded
    df = dataframes[main_rs]
    print(f"\nChosen record set for analysis: {main_rs}")
    print("Columns:", df.columns.tolist())
    display(df.head(5))
else:
    print("No tabular data found to proceed.")

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group data by fields using their `@id`s. This may include basic filtering, normalization, and aggregation.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if dataframes:
    # Use the selected dataframe and try to identify numeric columns
    df = dataframes[main_rs]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to coerce all columns to numeric and see what works
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        # Pick the first numeric column as example
        numeric_field_id = numeric_cols[0]
        print(f"Numeric field selected for EDA: {numeric_field_id}")

        # Filter by a threshold (10, for demonstration)
        threshold = 10
        if (df[numeric_field_id] > threshold).any():
            filtered_df = df[df[numeric_field_id] > threshold].copy()
        else:
            filtered_df = df.copy()  # No values satisfy threshold; take all rows

        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head(3))
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized values for '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(5))

        # Group by a categorical field, if available
        cat_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
        # skip columns with high cardinality or long text
        cat_candidates = [c for c in cat_candidates if df[c].nunique() < 20 and df[c].str.len().mean() < 20]
        if cat_candidates:
            group_field_id = cat_candidates[0]
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
            display(grouped.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize a numeric column's distribution and numeric field by group (if available), using only field/column `@id`s in axis labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_cols and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if cat_candidates and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Visualization skipped (no numeric field available).")

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 colorectal cancer survivor cohort dataset using the Croissant schema and the `mlcroissant` Python library. 

- Data was accessed using record set and field/column `@id`s for robust referencing.
- An initial EDA identified numeric fields, basic filters, and groupings.
- Visualizations illustrated data distributions and relationships.

For further analysis, consult the Croissant schema for deeper semantic field meanings and use additional processing and visualization as required for your research question.